In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

chatModel = ChatGroq(
    model="llama-3.3-70b-versatile"
)

c:\Users\DELL\miniconda3\envs\llmapp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
chatModel.invoke("What is the capital of France?")

AIMessage(content='The capital of France is Paris.', response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.012050263, 'completion_tokens_details': None, 'prompt_time': 0.001247259, 'prompt_tokens_details': None, 'queue_time': 0.161367299, 'total_time': 0.013297522}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'finish_reason': 'stop', 'logprobs': None}, id='run-7798c023-bd38-4ab7-bafa-217d64dfed80-0', usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50})

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [4]:
prompt=ChatPromptTemplate.from_template("tell me a curious fact about the {cricket_player}")
output_parser=StrOutputParser()

In [5]:
chain=prompt|chatModel|output_parser
chain.invoke({"cricket_player":"Virat Kohli"})

"Here's a curious fact about Virat Kohli: \n\nVirat Kohli's first cricket coach, Rajkumar Sharma, made him bat with a heavy bat and a few extra pounds of weight to improve his strength and technique. This unorthodox training method helped Kohli develop his trademark powerful strokeplay and endurance. Isn't that interesting?"

### Use of .bind() to add arguments to a Runnable in a LCEL Chain
    -For example,we can add an argument to stop the model response when it reaches the word "Virat"

In [7]:
chain=prompt|chatModel.bind(stop=["Kohli"])|output_parser
chain.invoke({"cricket_player":"Virat Kohli"})

"Here's one: \n\nVirat "

### Combining LCEL Chains

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser   

prompt=ChatPromptTemplate.from_template("tell me a curious fact about the {politican}")
chain=prompt|chatModel|output_parser
chain.invoke({"politican":"Narendra Modi"})

'One curious fact about Narendra Modi is that before he became the Prime Minister of India, he was a tea seller, also known as a \'chaiwala\', at a railway station in Gujarat. In fact, he has often spoken about his humble beginnings and has even referred to himself as a "chaiwala" in public speeches. This unique aspect of his background has been seen as a way to connect with the common people of India and has been a part of his populist appeal.'

### Combined chain

In [20]:
historian_prompt = ChatPromptTemplate.from_template("Was {politican} positive for Humanity?")

composed_chain = {"politican": chain} | historian_prompt | chatModel | StrOutputParser()

In [21]:
composed_chain.invoke({"politican":"jagan"})

"The phenomenon of the flag on top of the Jagannath Temple in Puri, India, flapping in the opposite direction of the wind is indeed a fascinating and intriguing fact. While it may not have a direct, tangible impact on humanity, it can be argued that it has a positive effect in several ways:\n\n1. **Inspiring curiosity and wonder**: The unexplained phenomenon sparks curiosity and encourages people to think about the natural world and the laws of physics. It inspires a sense of wonder and awe, which can be beneficial for individuals, particularly children, as it can foster a love for learning and exploration.\n2. **Promoting scientific inquiry**: The flag's behavior has puzzled scientists, and attempts to explain it have led to various investigations and research. This can contribute to the advancement of scientific knowledge, as scientists strive to understand the underlying mechanisms and principles that govern this phenomenon.\n3. **Cultural and tourist significance**: The Jagannath T

### Another example: a chain inside another chain

In [22]:

from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


prompt1 = ChatPromptTemplate.from_template("what is the country {politician} is from?")
prompt2 = ChatPromptTemplate.from_template(
    "what continent is the country {country} in? respond in {language}"
)


chain1 = prompt1 | chatModel | StrOutputParser()

chain2 = (
    {"country": chain1, "language": itemgetter("language")}
    | prompt2
    | chatModel
    | StrOutputParser()
)

chain2.invoke({"politician": "Miterrand", "language": "French"})

'Le pays associé à François Mitterrand est la France, et la France est située sur le continent européen.'

### LCEL chain at work in a typical RAG app

In [24]:
import bs4
from langchain import hub
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

splits = text_splitter.split_documents(docs)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
) 

vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

retriever = vectorstore.as_retriever()

prompt = hub.pull("rlm/rag-prompt")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | chatModel
    | StrOutputParser()
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
c:\Users\DELL\miniconda3\envs\llmapp\Lib\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [25]:

rag_chain.invoke("What is Task Decomposition?")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


'Task decomposition is the process of breaking down a complex task into smaller, simpler steps or subgoals. This can be done using various methods, including simple prompting, task-specific instructions, or with human inputs. It involves identifying the individual steps or components required to complete a task, allowing for more efficient planning and execution.'